# Text preprocessing for language models and classification models

In [1]:
%load_ext autoreload
%autoreload 2

%matplotlib inline

In [2]:
from lgm import *

In [3]:
# torch.cuda.get_device_name(torch.cuda.current_device())

In [4]:
path = untar_data(URLs.IMDB)

In [5]:
# path.ls()

We define a subclass of `ItemList`, called `TextList`, that will read the texts in the corresponding filenames.

- `ItemList` is basically just a container that has length and can be indexed / sliced into.
- `TextList` just adds a method `from_files` that recursively lists files with the `.txt` extension in a path and reads their contents

Just in case there are some text log files, we restrict the ones we take to the training, test, and unsupervised folders.

In [6]:
textlist = TextList.from_files(path, include=['train', 'test', 'unsup'])

We should expect a total of 100,000 texts.

In [7]:
len(textlist.items)

100000

Note the textlist contains just the paths to the files. The files are read "just in time" when we index/slice into the list.

In [8]:
# textlist

Here are the first 3 texts:

In [9]:
textlist[:3]

["I think the cards were stacked against Webmaster, because right from the start there was this itchy feeling, like something was wrong but I couldn't quite put my finger on it. Then it hit me. Dubbed. For a little while, they managed most of the lines either as voice over or off screen, with just a little hint here and there, until it became painfully obvious. This is the kind of dubbing that grates on the nerves, with nothing even remotely funny about it. I hate dubbing, but at least, however misplaced, martial arts films badly dubbed tend to have a sense of humour about it.<br /><br />What I wanted was a film about a hacker doing actual hacking and stuff like that. Maybe like a reverse side of the table of the movie Hackers (being about the person trying to keep them out instead about the people trying to get in). What I got was some poorly written, nonsensical at times murder mystery with a ton of bad chase sequences, a supposedly inept hacker who was neutered without his little eg

We split the data into a train set (90%) and a validation set (10%).

In [10]:
splitdata = SplitData.split_by_func(textlist, partial(random_splitter, proportion_valid=0.1))
splitdata

SplitData
Train: TextList (90012 items)
[/home/ady/Desktop/Dropbox/lgm_notebooks/data/imdb/test/neg/4225_2.txt, /home/ady/Desktop/Dropbox/lgm_notebooks/data/imdb/test/neg/9483_2.txt, /home/ady/Desktop/Dropbox/lgm_notebooks/data/imdb/test/neg/10083_2.txt, ...]
Path: /home/ady/Desktop/Dropbox/lgm_notebooks/data/imdb
Valid: TextList (9988 items)
[/home/ady/Desktop/Dropbox/lgm_notebooks/data/imdb/test/neg/6613_2.txt, /home/ady/Desktop/Dropbox/lgm_notebooks/data/imdb/test/neg/9339_3.txt, /home/ady/Desktop/Dropbox/lgm_notebooks/data/imdb/test/neg/7784_1.txt, ...]
Path: /home/ady/Desktop/Dropbox/lgm_notebooks/data/imdb

## Tokenizing

We need to tokenize the dataset first, which is splitting a sentence in individual tokens. Those tokens are the basic words or punctuation signs with some morphemic analysis -- "don't" is split into "do" and "n't" etc. We will use a processor for this, in conjunction with the [spacy library](https://spacy.io/).

Before tokenization, we clean up and process the texts in various ways:

- we remove HTML markup
- we identify character repetitions, replace them with only one occurence of the character and the number of repetitions
- we do the same for word repetitions.

After tokenization, we post-process the tokens a little more:

- we lower-case all-caps tokens and add a special token to mark that change
- we lower-case capitalization and mark that change,
- we add beginning-of-string BOS tokens; we will concatenate all texts and stream the result, so BOS tokens will mark the original text boundaries for the neural network model

Since tokenizing and applying the rules before and after tokenization takes a bit of time, we'll parallelize all this using `ProcessPoolExecutor` to go faster.

In [11]:
proc_tok = TokenizeProcessor(max_workers=4)

## Numericalizing

Once we have tokenized our texts, we replace each token by an individual number, i.e., we numericalize the texts. Again, we do this with a processor.

When we do language modeling, we will infer the labels from the text during training, so there's no need to label. The training loop expects labels however, so we need to add dummy ones.

In [12]:
proc_num = NumericalizeProcessor()

## Actually processing the text

The `label_by_func` function from the `lgm` library takes a splitdata object, i.e., a dataset split into train and validation, and:

- labels each of them by a function -- a dummy function in this case, which labels everything with $0$;
- processes each of them with the provided processors; in this case, we tokenize and numericalize the indepedent variable -- hence processor_x; processor_y would list processors for the dependent variable
- reassembles the labeled and processed train and validation sets into a new SplitData object

In [13]:
# %time labeled_list = label_by_func(splitdata, lambda x: 0, processor_x = [proc_tok, proc_num])

CPU times: user 13.8 s, sys: 1.98 s, total: 15.8 s
Wall time: 57.2 s


Once the texts have been processed, they become lists of numbers, but we can still access the underlying raw data in `x_obj` (or `y_obj` for the targets, but we don't have any here).

In [14]:
labeled_list.train

LabeledData
x: TextList (90012 items)
[[2, 18, 122, 8, 4051, 86, 17559, 467, 7, 48211, 10, 106, 229, 51, 8, 392, 54, 25, 19, 19652, 573, 10, 53, 158, 25, 379, 30, 18, 95, 35, 205, 301, 77, 3685, 34, 16, 9, 7, 115, 16, 614, 89, 9, 7, 2324, 9, 7, 28, 12, 139, 153, 10, 46, 1336, 110, 13, 8, 431, 372, 26, 584, 143, 55, 141, 278, 10, 27, 57, 12, 139, 2882, 148, 11, 54, 10, 383, 16, 900, 2290, 606, 9, 7, 19, 15, 8, 266, 13, 3507, 20, 17094, 34, 8, 5714, 10, 27, 178, 76, 2604, 171, 59, 16, 9, 18, 736, 3507, 10, 30, 45, 243, 10, 216, 8588, 10, 1618, 1653, 124, 953, 2324, 2440, 14, 41, 12, 291, 13, 1175, 59, 16, 9, 24, 7, 64, 18, 488, 25, 12, 31, 59, 12, 11985, 423, 868, 12073, 11, 539, 53, 20, 9, 7, 298, 53, 12, 7404, 510, 13, 8, 2337, 13, 8, 29, 7, 14778, 36, 129, 59, 8, 422, 282, 14, 409, 111, 61, 319, 59, 8, 100, 282, 14, 99, 17, 33, 9, 7, 64, 18, 209, 25, 65, 859, 433, 10, 5147, 45, 234, 590, 778, 27, 12, 6604, 13, 97, 1245, 834, 10, 12, 1500, 2792, 11985, 49, 25, 26211, 230, 40, 139, 3854

In [15]:
labeled_list.train.x_obj(0)

"_BOS_ i think the cards were stacked against _CAP_ webmaster , because right from the start there was this itchy feeling , like something was wrong but i could n't quite put my finger on it . _CAP_ then it hit me . _CAP_ dubbed . _CAP_ for a little while , they managed most of the lines either as voice over or off screen , with just a little hint here and there , until it became painfully obvious . _CAP_ this is the kind of dubbing that grates on the nerves , with nothing even remotely funny about it . i hate dubbing , but at least , however misplaced , martial arts films badly dubbed tend to have a sense of humour about it . \n\n _CAP_ what i wanted was a film about a hacker doing actual hacking and stuff like that . _CAP_ maybe like a reverse side of the table of the movie _CAP_ hackers ( being about the person trying to keep them out instead about the people trying to get in ) . _CAP_ what i got was some poorly written , nonsensical at times murder mystery with a ton of bad chase s

In [16]:
labeled_list.train.x_obj(47)

"_BOS_ _CAP_ _UNK_ _CAP_ ivory _CAP_ wayans was so funny in _CAP_ low _CAP_ down _CAP_ dirty _CAP_ shame that i had to see this one and it was one of the worst he has done and _CAP_ steven _CAP_ seagal did n't help much . _CAP_ it starts off with some odd religious killings that do n't make much sense to _CAP_ jim _CAP_ campbell ( _CAP_ keenan ) . _CAP_ he is surprised to see a new partner waiting for him to work by his side to crack the case but _CAP_ jack _CAP_ cole does n't seem to be who everyone thinks he is until _CAP_ jack 's ex wife is killed in one of those ritual killings that end up making him the suspect as well . _CAP_ it 's the same thing as all of his other movies : _CAP_ smoke past , cia involvement and now trying to be a normal cop . _CAP_ why does _CAP_ steven dress up like he is from a _CAP_ western movie ? _CAP_ and the prayer _UNK_ on top of that make things a little confusing ."

In [17]:
labeled_list.valid

LabeledData
x: TextList (9988 items)
[[2, 7, 19, 29, 261, 486, 53, 12, 101, 338, 17, 1832, 23, 391, 9, 21, 7, 306, 22, 114, 12, 29, 59, 42, 13, 8, 857, 11, 110, 3266, 8293, 19142, 13, 8, 702, 962, 50, 7, 11, 306, 22, 196, 7, 1928, 7, 0, 26, 7, 3023, 7, 3248, 7, 3740, 50, 21, 7, 20, 22, 135, 19, 29, 436, 1999, 379, 9, 7, 155, 196, 48, 303, 49, 3225, 74, 8167, 13, 8, 145, 39, 22, 2382, 66, 7, 11, 115, 10, 155, 306, 19, 303, 494, 40, 123, 104, 37, 7, 3023, 7, 3740, 10, 30, 7, 1928, 7, 0, 17, 12, 811, 16369, 66, 7, 26, 18, 1846, 168, 19, 29, 34, 27852, 10, 18, 87, 35, 186, 564, 3759, 19, 145, 25, 177, 7, 3023, 7, 3740, 9, 7, 39, 87, 35, 185, 53, 108, 10, 706, 53, 108, 10, 506, 53, 108, 10, 55, 76, 1242, 53, 108, 9, 18, 95, 37, 99, 521, 19, 212, 10, 11, 40424, 10, 18, 95, 37, 390, 8, 29, 9, 7, 68, 7, 863, 7, 5222, 11, 7, 613, 7, 7457, 86, 196, 26, 8, 11539, 7, 9444, 7, 7146, 11, 8, 7, 6711, 7, 542, 10, 90, 87, 35, 477, 63, 46, 86, 1844, 1313, 4145, 13, 79, 316, 121, 106, 110, 13, 200, 85, 1

In [18]:
labeled_list.valid.x_obj(5)

"_BOS_ _CAP_ congo is another multi - _UNK_ dollar adaptation of _CAP_ crichton 's works . _CAP_ like _CAP_ jurassic _CAP_ park , _CAP_ the _CAP_ lost _CAP_ world , _CAP_ sphere , etc , the film raped the book of its true meaning and essence . i 'll make this short and to the point . _CAP_ the scenery is beautiful . _CAP_ the actors , well it 's the best they can do . _CAP_ the script ? _CAP_ try _UNK_ hundreds of pages into an hour and half movie . _CAP_ you get a mess in the end but how neat of a mess is what counts and _CAP_ congo falls somewhere below that . _CAP_ there were some silly moments , like why did the killer gorillas decide to jump into the lava ? _CAP_ and _CAP_ amy , raised by humans , surrounded by humans , yet can intimidate dozens of killer apes around her ? _CAP_ what sort of twist of common sense is that ? _CAP_ which brings me to this . _CAP_ if there was an annoying character in every movie , _CAP_ amy ranks of one here . _CAP_ you see _CAP_ amy is this naive li

We can also convert numericalized text back by using the `deprocess` method associated with the numericalization processor `proc_num`.

In [19]:
print(proc_num.deprocess(labeled_list.train[0][0]))

['_BOS_', 'i', 'think', 'the', 'cards', 'were', 'stacked', 'against', '_CAP_', 'webmaster', ',', 'because', 'right', 'from', 'the', 'start', 'there', 'was', 'this', 'itchy', 'feeling', ',', 'like', 'something', 'was', 'wrong', 'but', 'i', 'could', "n't", 'quite', 'put', 'my', 'finger', 'on', 'it', '.', '_CAP_', 'then', 'it', 'hit', 'me', '.', '_CAP_', 'dubbed', '.', '_CAP_', 'for', 'a', 'little', 'while', ',', 'they', 'managed', 'most', 'of', 'the', 'lines', 'either', 'as', 'voice', 'over', 'or', 'off', 'screen', ',', 'with', 'just', 'a', 'little', 'hint', 'here', 'and', 'there', ',', 'until', 'it', 'became', 'painfully', 'obvious', '.', '_CAP_', 'this', 'is', 'the', 'kind', 'of', 'dubbing', 'that', 'grates', 'on', 'the', 'nerves', ',', 'with', 'nothing', 'even', 'remotely', 'funny', 'about', 'it', '.', 'i', 'hate', 'dubbing', ',', 'but', 'at', 'least', ',', 'however', 'misplaced', ',', 'martial', 'arts', 'films', 'badly', 'dubbed', 'tend', 'to', 'have', 'a', 'sense', 'of', 'humour', '

Compare with the original text (assuming the first text made it into train, not validation):

In [20]:
print(textlist[0])

I think the cards were stacked against Webmaster, because right from the start there was this itchy feeling, like something was wrong but I couldn't quite put my finger on it. Then it hit me. Dubbed. For a little while, they managed most of the lines either as voice over or off screen, with just a little hint here and there, until it became painfully obvious. This is the kind of dubbing that grates on the nerves, with nothing even remotely funny about it. I hate dubbing, but at least, however misplaced, martial arts films badly dubbed tend to have a sense of humour about it.<br /><br />What I wanted was a film about a hacker doing actual hacking and stuff like that. Maybe like a reverse side of the table of the movie Hackers (being about the person trying to keep them out instead about the people trying to get in). What I got was some poorly written, nonsensical at times murder mystery with a ton of bad chase sequences, a supposedly inept hacker who was neutered without his little ego,

Since the preprocessing takes time, we save the intermediate result using pickle. Don't use any lambda functions in your processors or they won't be able to pickle.

In [21]:
import pickle
# pickle.dump(labeled_list, open(path/'labeled_list_lm.pkl', 'wb'))

In [22]:
labeled_list = pickle.load(open(path/'labeled_list_lm.pkl', 'rb'))

## Batching for language models

Convert our `labeled_list` to a `DataBunch`, that is, batchifying it through a dataloader requires additional work.

- we don't just want batches of IMDB reviews: we want to stream in _all the texts concatenated_ and determine batches based on that single concatenated text stream + the BPTT (backprop through time) hyperparam.
- we also have to prepare the targets that are the next words in the text.

This is done with an `LM_PreLoader` class, which processes texts for language models (hence `LM`) so that they can be fed into dataloaders (hence `PreLoader`):

- at the beginning of each epoch, we shuffle the texts (if `shuffle=True`) and create a big stream by concatenating all of them
- we divide this big stream in `batch_size` smaller streams
- we read these smaller streams in chunks of bptt length

Let's see how this works for the smaller validation set:

In [23]:
dl = DataLoader(LM_PreLoader(labeled_list.valid, shuffle=True), batch_size=64)

Let's check it all works ok: `x1`, `y1`, `x2` and `y2` should all be of size `batch_size`  by `bptt`. The texts in each row of `x1` should continue in `x2`. `y1` and `y2` should have the same texts as their `x` counterpart, shifted of one position to the right.

In [24]:
iter_dl = iter(dl)
x1,y1 = next(iter_dl)
x2,y2 = next(iter_dl)

In [25]:
x1.size(),y1.size()

(torch.Size([64, 70]), torch.Size([64, 70]))

In [26]:
print(proc_num.deprocess(x1[0]))

['_BOS_', '_CAP_', 'this', 'movie', 'twists', 'the', 'facts', 'of', '_CAP_', 'anne', 'and', '_CAP_', 'mary', "'s", 'lives', 'into', 'something', 'unrecognizable', '.', '_CAP_', 'to', 'make', '_CAP_', 'mary', '_CAP_', 'boleyn', ',', 'who', 'in', 'fact', 'was', 'a', 'rather', 'dim', 'and', 'foolish', 'creature', ',', 'and', 'make', 'her', 'the', '"', 'good', '"', 'sister', 'is', 'just', 'silly', '.', '_CAP_', 'it', 'is', '_CAP_', 'anne', 'who', 'was', 'in', 'fact', 'the', 'far', 'more', 'interesting', 'character', ',', 'and', 'that', 'is', 'why', 'it']


In [27]:
print(proc_num.deprocess(y1[0]))

['_CAP_', 'this', 'movie', 'twists', 'the', 'facts', 'of', '_CAP_', 'anne', 'and', '_CAP_', 'mary', "'s", 'lives', 'into', 'something', 'unrecognizable', '.', '_CAP_', 'to', 'make', '_CAP_', 'mary', '_CAP_', 'boleyn', ',', 'who', 'in', 'fact', 'was', 'a', 'rather', 'dim', 'and', 'foolish', 'creature', ',', 'and', 'make', 'her', 'the', '"', 'good', '"', 'sister', 'is', 'just', 'silly', '.', '_CAP_', 'it', 'is', '_CAP_', 'anne', 'who', 'was', 'in', 'fact', 'the', 'far', 'more', 'interesting', 'character', ',', 'and', 'that', 'is', 'why', 'it', 'is']


In [28]:
print(proc_num.deprocess(x2[0]))

['is', 'her', 'life', ',', 'and', 'not', '_CAP_', 'mary', "'s", ',', 'that', 'has', 'been', 'told', 'so', 'often', '.', '\n\n', '_CAP_', 'in', 'response', 'to', 'an', 'earlier', 'review', ',', 'i', 'fail', 'to', 'see', 'how', '_CAP_', 'anne', "'s", 'life', 'was', 'so', '"', 'criminal', '"', '...', 'to', 'me', 'it', "'s", '_CAP_', 'henry', 'who', 'was', 'the', 'real', 'criminal', '.', '_CAP_', 'whatever', '_CAP_', 'anne', "'s", 'motives', 'for', 'winning', 'the', 'king', 'and', 'withholding', 'her', 'affections', 'in', 'order', 'to']


In [29]:
print(proc_num.deprocess(y2[0]))

['her', 'life', ',', 'and', 'not', '_CAP_', 'mary', "'s", ',', 'that', 'has', 'been', 'told', 'so', 'often', '.', '\n\n', '_CAP_', 'in', 'response', 'to', 'an', 'earlier', 'review', ',', 'i', 'fail', 'to', 'see', 'how', '_CAP_', 'anne', "'s", 'life', 'was', 'so', '"', 'criminal', '"', '...', 'to', 'me', 'it', "'s", '_CAP_', 'henry', 'who', 'was', 'the', 'real', 'criminal', '.', '_CAP_', 'whatever', '_CAP_', 'anne', "'s", 'motives', 'for', 'winning', 'the', 'king', 'and', 'withholding', 'her', 'affections', 'in', 'order', 'to', 'gain']


Let's use a convenience function to do this quickly, for both the train and validation datasets.

In [30]:
# def get_lm_dls(train_ds, valid_ds, batch_size, bptt, **kwargs):
#     return (DataLoader(LM_PreLoader(train_ds, batch_size, bptt, shuffle=True),
#                        batch_size=batch_size, **kwargs),
#             DataLoader(LM_PreLoader(valid_ds, batch_size, bptt, shuffle=False),
#                        batch_size=2*batch_size, **kwargs))

# def lm_databunchify(splitdata, batch_size, bptt, **kwargs):
#     return DataBunch(*get_lm_dls(splitdata.train, splitdata.valid, batch_size, bptt, **kwargs))

In [31]:
batch_size = 64
bptt = 70
data = lm_databunchify(labeled_list, batch_size, bptt)

Optional: check out the doc string for the `__getitem__` method of `LM_PreLoader` for a simple example of how batching for language models works.

In [32]:
# LM_PreLoader.__getitem__?

## Batching for classification

When we will want to tackle classification, gathering the data will be a bit different: first we will label our texts with the folder they come from, and then we will need to apply padding to batch them together. To avoid mixing very long texts with very short ones, we will also use `Sampler` to sort our samples by length, with a bit of randomness for the training set.

We label our data with `CategoryProcessor`, which converts the labels (levels of the dependent categorical variable) to integers.

In [33]:
proc_cat = CategoryProcessor()

In [34]:
textlist = TextList.from_files(path, include=['train', 'test'])
splitdata = SplitData.split_by_func(textlist, partial(grandparent_splitter, valid_name='test'))
labeled_list = label_by_func(splitdata, parent_labeler,
                             processor_x = [proc_tok, proc_num],
                             processor_y=proc_cat)

In [35]:
proc_cat.level_dict

['neg', 'pos']

In [36]:
proc_cat.otoi

{'neg': 0, 'pos': 1}

In [37]:
labeled_list.train

LabeledData
x: TextList (25000 items)
[[2, 7, 19, 29, 15, 175, 7, 1613, 2312, 31, 17, 8, 369, 13, 7, 8, 7, 10530, 7, 2589, 9, 7, 37, 20, 20, 15, 2773, 97, 30, 28, 8, 212, 20, 110, 2312, 124, 3609, 7313, 11, 1798, 28, 8, 760, 46, 658, 14, 1646, 9, 7, 63, 32, 390, 12, 24194, 1132, 13, 136, 34, 8, 1788, 11, 8, 116, 8, 7, 11076, 78, 677, 12, 136, 10, 93, 402, 32, 227, 390, 19, 29, 9, 18, 157, 10, 597, 147, 308, 11, 824, 7, 8, 7, 1524, 11, 7, 8, 7, 24863, 55, 7, 8, 7, 1736, 9, 7, 68, 105, 7, 1613, 989, 865, 20, 558, 100, 157, 97, 690, 66, 7, 16, 25, 4593, 14, 84, 2805, 2403, 49, 38, 37, 1514, 14, 3947, 36, 3730, 66, 2805, 157, 97, 690, 66, 33, 11, 1068, 121, 18, 83, 95, 37, 2025, 14, 9, 7, 103, 10, 16, 73, 161, 98, 101, 63, 8, 29, 85, 636, 65, 34907, 7, 166, 20, 73, 43, 158, 18, 321, 53, 14, 968, 14, 84, 9, 7, 49, 25, 8, 46050, 49, 1129, 19, 16754, 21245, 23, 1596, 82, 13, 12, 29, 27, 7, 8, 7, 21651, 7, 9395, 11, 7, 8, 7, 10312, 66], [2, 7, 83, 12, 410, 29, 9, 7, 16, 22, 14, 43, 847, 10, 17

In [38]:
labeled_list.train.x_obj(0)

"_BOS_ _CAP_ this movie is another _CAP_ christian propaganda film in the line of _CAP_ the _CAP_ omega _CAP_ code . _CAP_ not that that is necessarily bad but for the fact that most propaganda films sacrifice sincerity and realism for the message they wish to deliver . _CAP_ if you enjoy a styrofoam portrayal of life on the streets and the way the _CAP_ gospel can change a life , than perhaps you may enjoy this movie . i say , save your money and rent _CAP_ the _CAP_ cross and _CAP_ the _CAP_ switchblade or _CAP_ the _CAP_ mission . _CAP_ when will _CAP_ christian directors learn that sometimes people say bad words ? _CAP_ it was frustrating to see criminals depicted who are not allowed to swear ( huh ? criminals say bad words ? ) and flat characters i really could not relate to . _CAP_ also , it would 've been great if the movie had shown some t&a. _CAP_ now that would be something i 'd like to pay to see . _CAP_ who was the blockhead who compared this communion wafer - thin story of

In [39]:
labeled_list.train.x_obj(47)

'_BOS_ _CAP_ now please do n\'t start calling me names like , " unpatriotic " , " weirdo " and more . \n\n _CAP_ the very length of this movie ( 4 hours .. ! ! ! ) is its biggest mistake . _CAP_ no editing at all - seems like j.p. _CAP_ dutta fell in love with his project too much . _CAP_ even _CAP_ lagaan was 4 hours long - but it was entertaining and gave a message as well . \n\n _CAP_ it \'s based on true incidents and real people . _CAP_ kudos to it , but were the repetitive war scenes really needed ? _CAP_ on top of it the focus constantly shifted from one battalion / squadron to another and it was impossible to keep a track of them all . \n\n _CAP_ between the skirmishes , there were songs about loneliness , _UNK_ and related stuff . _CAP_ there were chummy conversations . _CAP_ in the beginning it gave some relief from the violence but became so monotonous later that one could even correctly predict nature of the forthcoming talk . \n\n _CAP_ why were the soldiers walking around

In [40]:
labeled_list.valid

LabeledData
x: TextList (25000 items)
[[2, 18, 122, 8, 4051, 86, 17559, 467, 7, 48211, 10, 106, 229, 51, 8, 392, 54, 25, 19, 19652, 573, 10, 53, 158, 25, 379, 30, 18, 95, 35, 205, 301, 77, 3685, 34, 16, 9, 7, 115, 16, 614, 89, 9, 7, 2324, 9, 7, 28, 12, 139, 153, 10, 46, 1336, 110, 13, 8, 431, 372, 26, 584, 143, 55, 141, 278, 10, 27, 57, 12, 139, 2882, 148, 11, 54, 10, 383, 16, 900, 2290, 606, 9, 7, 19, 15, 8, 266, 13, 3507, 20, 17094, 34, 8, 5714, 10, 27, 178, 76, 2604, 171, 59, 16, 9, 18, 736, 3507, 10, 30, 45, 243, 10, 216, 8588, 10, 1618, 1653, 124, 953, 2324, 2440, 14, 41, 12, 291, 13, 1175, 59, 16, 9, 24, 7, 64, 18, 488, 25, 12, 31, 59, 12, 11985, 423, 868, 12073, 11, 539, 53, 20, 9, 7, 298, 53, 12, 7404, 510, 13, 8, 2337, 13, 8, 29, 7, 14778, 36, 129, 59, 8, 422, 282, 14, 409, 111, 61, 319, 59, 8, 100, 282, 14, 99, 17, 33, 9, 7, 64, 18, 209, 25, 65, 859, 433, 10, 5147, 45, 234, 590, 778, 27, 12, 6604, 13, 97, 1245, 834, 10, 12, 1500, 2792, 11985, 49, 25, 26211, 230, 40, 139, 3854

In [41]:
labeled_list.valid.x_obj(5)

'_BOS_ _CAP_ this film truly was poor . i went to the theatre expecting something exciting , and instead was afforded the opportunity to hone my " guess the next plot twist before it happens " skills . _CAP_ seriously , the plot was written with an extra thick crayon so everyone could see . _CAP_ nothing was truly shocking . _CAP_ in fact , even the gore was met with such complete suspension of belief that it really did n\'t add up to much . \n\n _CAP_ the excessive wise cracking and cops talking shop at the crime scenes made it seem all the more phony . _CAP_ and the scene where _CAP_ lambert \'s character is struggling with the clues and reaches his " investigative epiphany " goes to great lengths to indicate the level of intellect expected from the audience - little . \n\n _CAP_ probably the most annoying aspect of the cinematography was the " x - _CAP_ files " treatment : _CAP_ every building in the film , whether it \'s the precinct building , or a house at noon , or a hospital , 

In [42]:
# import pickle
# pickle.dump(labeled_list, open(path/'labeled_list_clas.pkl', 'wb'))

In [43]:
labeled_list = pickle.load(open(path/'labeled_list_clas.pkl', 'rb'))

Let's check that the labels seem consistent with the texts.

In [44]:
[(labeled_list.train.x_obj(i), labeled_list.train.y_obj(i)) for i in [10, 732, 12552]]

[('_BOS_ _CAP_ gee , what a crappy movie this was ! i can not understand what people find so scary about " _CAP_ the _CAP_ grudge " . _CAP_ the director plays one trick ( i \'d have to admit a very good one , that is brought to life very stylized ) and then he repeats it for the rest of the movie over and over again . _CAP_ as a consequence i startled a few times in the first quarter of the movie , but once i knew the drill i practically fell asleep as _CAP_ the _CAP_ grudge grew more and more predictable by the minute . _CAP_ to conclude , i can say that there are a lot better movies in the genre to begin with , that the so - called predecessor " _CAP_ the _CAP_ ring " was way scarier and that buying a ticket for " _CAP_ the _CAP_ grudge " is a waste of money .',
  'neg'),
 ('_BOS_ because you can put it on fast forward and watch the inane story , without having to listen to banal dialogue , and be finished in 10 minutes max . _CAP_ come to think of it , even 10 minutes is too much to

For the validation set, we will simply sort the samples by length, and we begin with the longest ones for memory reasons (it's better to always have the biggest tensors first).

For the training set, we want some kind of randomness on top of this:

- we shuffle the texts and build megabatches of size `50 * batch_size`
- we sort those megabatches by length before splitting them in 50 minibatches; that way we will have randomized batches of roughly the same length
- then we make sure to have the biggest batch first and shuffle the order of the other batches
- we also make sure the last batch stays at the end because its size is probably lower than batch_size

Padding: we add the padding token (id of 1) at the end of each sequence to make them all the same size when batching them. Note that we need padding _at the end_ to be able to use `PyTorch` convenience functions that will let us ignore that padding (more on this later in the ULMFiT notebook).

In [45]:
# def pad_collate(samples, pad_idx=1, pad_first=False):
#     # identify the longest document in the minibatch
#     max_len = max([len(sample[0]) for sample in samples])
#     # create rectangular tensor that can accommodate all documents
#     # in the batch up to that max_len, and fill it with padding.
#     results = torch.zeros(len(samples), max_len).long() + pad_idx
#     # take documents in the minibatch and put them in the tensor
#     # keeping padding either at the beginning or at the end
#     for i, sample in enumerate(samples):
#         if pad_first:
#             results[i, -len(sample[0]):] = LongTensor(sample[0])
#         else:
#             results[i, :len(sample[0]) ] = LongTensor(sample[0])
#     return results, tensor([sample[1] for sample in samples])

In [46]:
batch_size = 64
train_sampler = SortishSampler(labeled_list.train.x,
                               key=lambda t: len(labeled_list.train[int(t)][0]),
                               batch_size=batch_size)
train_dl = DataLoader(labeled_list.train, batch_size=batch_size,
                      sampler=train_sampler, collate_fn=pad_collate)

Let's look at one training batch:

In [47]:
iter_dl = iter(train_dl)
x, y = next(iter_dl)

We can see the padding at the end of the non-initial movie reviews:

In [48]:
x

tensor([[    2,     7,  1148,  ...,    12, 15754,    24],
        [    2,     7,  4521,  ...,     1,     1,     1],
        [    2,     7,    65,  ...,     1,     1,     1],
        ...,
        [    2,    19,  1423,  ...,     1,     1,     1],
        [    2,     7,   175,  ...,     1,     1,     1],
        [    2,   402,    17,  ...,     1,     1,     1]])

In [49]:
x.size()

torch.Size([64, 3310])

In [50]:
y

tensor([1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1,
        1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1,
        1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1])

In [51]:
y.size()

torch.Size([64])

Let's look at the lengths of the documents in this batch. We can get their length by subtracting the number of padding tokens from the length of the tensor:

In [52]:
lengths = []
for i in range(x.size(0)):
    lengths.append(x.size(1) - (x[i]==1).sum().item())
print(lengths)

[3310, 2211, 1950, 1868, 1613, 1428, 1390, 1351, 1350, 1325, 1322, 1318, 1317, 1311, 1304, 1302, 1301, 1291, 1289, 1284, 1263, 1258, 1250, 1243, 1236, 1231, 1220, 1220, 1217, 1188, 1186, 1181, 1179, 1169, 1161, 1160, 1151, 1138, 1137, 1129, 1127, 1126, 1124, 1117, 1112, 1109, 1092, 1087, 1082, 1078, 1076, 1068, 1062, 1062, 1061, 1061, 1054, 1049, 1049, 1034, 1025, 1024, 1021, 1016]


This is the first batch so it has the longest movie review first. The last one is the shortest movie review in the batch.

If we look at the next batch, we see the lengths fall within a much narrower range:

In [53]:
x,y = next(iter_dl)
lengths = []
for i in range(x.size(0)):
    lengths.append(x.size(1) - (x[i]==1).sum().item())
print(lengths)

[359, 359, 358, 358, 357, 357, 356, 356, 356, 355, 355, 355, 355, 355, 355, 354, 354, 353, 353, 352, 352, 351, 351, 350, 350, 350, 350, 350, 349, 349, 349, 349, 348, 348, 347, 347, 347, 347, 346, 346, 345, 345, 344, 344, 344, 344, 343, 343, 342, 342, 342, 342, 342, 341, 340, 340, 340, 340, 339, 339, 339, 339, 339, 339]


And we add a convenience function:

In [54]:
# def get_clas_dls(train_ds, valid_ds, batch_size, **kwargs):
#     train_sampler = SortishSampler(train_ds.x,
#                                    key=lambda t: len(train_ds.x[t]),
#                                    batch_size=batch_size)
#     valid_sampler = SortSampler(valid_ds.x,
#                                 key=lambda t: len(valid_ds.x[t]))
#     return (DataLoader(train_ds, batch_size=batch_size, sampler=train_sampler,
#                        collate_fn=pad_collate, **kwargs),
#             DataLoader(valid_ds, batch_size=batch_size*2, sampler=valid_sampler,
#                        collate_fn=pad_collate, **kwargs))

# def clas_databunchify(splitdata, batch_size, **kwargs):
#     return DataBunch(*get_clas_dls(splitdata.train, splitdata.valid, batch_size, **kwargs))

In [55]:
batch_size = 64
bptt = 70
data = clas_databunchify(labeled_list, batch_size)